# 04 Classify Toxicity

Phase 4c: batch toxicity classification for Emberline Studios chat messages using `ai_query` on `system.ai.meta-llama-3-3-70b-instruct`. Classifies 7,000 reported messages (evidence from player reports) + 3,000 random unreported messages (10,000 total cap). Structured output: `category`, `severity`, `rationale`. Evaluated against ground-truth labels. Scores written to `ml_toxicity_message_labels`.

In [0]:
from pyspark.sql import functions as F

CATALOG  = "mg_aws_serverless_workspace_catalog"
SCHEMA   = "game_cx_players"
FQ       = f"{CATALOG}.{SCHEMA}"
AS_OF_DATE = "2026-12-31"
MODEL    = "system.ai.meta-llama-3-3-70b-instruct"
EVAL_MODEL = "system.ai.claude-opus-4-7"   # used only for the keyword-baseline eval commentary
N_REPORTED   = 7_000
N_UNREPORTED = 3_000
GT       = f"/Volumes/{CATALOG}/{SCHEMA}/ground_truth"
EXPERIMENT = "/Users/mayur.gupta@databricks.com/game-cx-players-toxicity-eval"
print(FQ, MODEL)

In [0]:
# Reported messages: evidence_message_id from player reports (the most likely toxic ones)
reported_ids = (spark.table(f"{FQ}.silver_player_reports")
                .select(F.col("evidence_message_id").alias("message_id"))
                .where("message_id IS NOT NULL").distinct())

chat = spark.table(f"{FQ}.silver_chat_messages").select("message_id", "message_text", "sender_player_id", "sent_date")

reported = (chat.join(reported_ids, "message_id")
            .withColumn("sample_type", F.lit("reported"))
            .orderBy(F.rand(seed=42)).limit(N_REPORTED))

unreported = (chat.join(reported_ids, "message_id", "left_anti")
              .withColumn("sample_type", F.lit("unreported"))
              .orderBy(F.rand(seed=42)).limit(N_UNREPORTED))

sample = reported.unionAll(unreported)
# Persist so we call ai_query exactly once per message (avoid calling twice on same input)
sample.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.ml_toxicity_sample")
print(spark.table(f"{FQ}.ml_toxicity_sample").groupBy("sample_type").count().collect())

In [0]:
# Simple keyword baseline for comparison — flag messages containing any of these tokens.
# Establishes the false-positive rate on BANTER we expect the LLM to beat.
KEYWORDS = ["kill", "die", "dead", "noob", "idiot", "stupid", "hate", "trash", "loser", "suck"]
pattern = "|".join(KEYWORDS)
spark.sql(f"""
  SELECT
    sample_type,
    COUNT(*) AS messages,
    SUM(CAST(LOWER(message_text) RLIKE '{pattern}' AS INT)) AS keyword_flagged,
    ROUND(SUM(CAST(LOWER(message_text) RLIKE '{pattern}' AS INT)) * 1.0 / COUNT(*), 4) AS keyword_flag_rate
  FROM {FQ}.ml_toxicity_sample
  GROUP BY ALL ORDER BY sample_type
""").show()

In [0]:
%sql
-- Batch classify all 10,000 sampled messages in one SQL pass.
-- ai_query is called once per row; results are stored immediately into ml_toxicity_message_labels.
-- Using failOnError => false so a single bad row doesn't abort the full batch.
CREATE OR REPLACE TABLE mg_aws_serverless_workspace_catalog.game_cx_players.ml_toxicity_message_labels
COMMENT 'Toxicity classification of 10,000 chat messages (7K reported + 3K random) from ai_query on system.ai.meta-llama-3-3-70b-instruct. One row per message.'
AS
SELECT
  message_id,
  sender_player_id,
  sent_date,
  sample_type,
  message_text,
  ai_query(
    'system.ai.meta-llama-3-3-70b-instruct',
    'You are a trust & safety classifier for a gaming platform. Classify the chat message below.\n\nRespond with a JSON object with these exact keys:\n- category: one of NEUTRAL, BANTER, HARASSMENT, HATE_SPEECH, THREAT, SCAM\n- severity: integer 0 (not toxic) to 3 (severe)\n- rationale: one sentence explaining the classification\n\nMessage: ' || message_text,
    responseFormat => '{"type":"json_schema","json_schema":{"name":"toxicity","schema":{"type":"object","properties":{"category":{"type":"string"},"severity":{"type":"integer"},"rationale":{"type":"string"}},"required":["category","severity","rationale"]}}}',
    failOnError => false
  ) AS llm_result
FROM mg_aws_serverless_workspace_catalog.game_cx_players.ml_toxicity_sample

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

result = spark.table(f"{FQ}.ml_toxicity_message_labels")

# failOnError => false with json_schema responseFormat returns STRUCT<result STRING, errorMessage STRING>
# Parse the JSON string in the result field
json_schema = StructType([
    StructField("category",  StringType(),  True),
    StructField("severity",  IntegerType(), True),
    StructField("rationale", StringType(),  True),
])
result = (result
  .withColumn("_parsed",   F.from_json(F.col("llm_result.result"), json_schema))
  .withColumn("category",  F.col("_parsed.category"))
  .withColumn("severity",  F.col("_parsed.severity"))
  .withColumn("rationale", F.col("_parsed.rationale"))
  .withColumn("llm_error", F.col("llm_result.errorMessage"))
  .withColumn("is_toxic",  F.col("category").isin("HARASSMENT", "HATE_SPEECH", "THREAT", "SCAM"))
  .drop("llm_result", "_parsed"))

result.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.ml_toxicity_message_labels")
result = spark.table(f"{FQ}.ml_toxicity_message_labels")
display(result.groupBy("category", "is_toxic").count().orderBy("is_toxic", "count", ascending=[False, False]))
print(f"Errors: {result.where('llm_error IS NOT NULL').count()}")

In [0]:
import mlflow, pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

gt = spark.read.parquet(f"{GT}/gt_chat_labels").select(
    F.col("message_id"), F.col("category").alias("gt_category"),
    F.col("is_toxic").alias("gt_is_toxic"), F.col("is_banter"))

eval_df = (result.join(gt, "message_id")
           .select("message_id", "sample_type", "category", "is_toxic",
                   "gt_category", "gt_is_toxic", "is_banter", "message_text"))

pdf = eval_df.toPandas()
n_matched = len(pdf)
print(f"Matched {n_matched} of 10,000 to ground truth")

# Binary toxic / not-toxic
kw_pattern = "kill|die|dead|noob|idiot|stupid|hate|trash|loser|suck"
pdf["kw_flag"] = pdf["message_text"].str.lower().str.contains(kw_pattern, regex=True)

from sklearn.metrics import precision_score, recall_score, f1_score
metrics = {}
for name, pred in [("llm", pdf["is_toxic"]), ("keyword", pdf["kw_flag"])]:
    metrics[f"{name}_precision"] = precision_score(pdf["gt_is_toxic"], pred, zero_division=0)
    metrics[f"{name}_recall"]    = recall_score(pdf["gt_is_toxic"], pred, zero_division=0)
    metrics[f"{name}_f1"]        = f1_score(pdf["gt_is_toxic"], pred, zero_division=0)

# Keyword false-positive rate on banter
banter = pdf[pdf["is_banter"] == True]
metrics["keyword_banter_fp_rate"] = float((banter["kw_flag"] == True).mean()) if len(banter) else 0.0
metrics["llm_banter_fp_rate"]     = float((banter["is_toxic"] == True).mean()) if len(banter) else 0.0
metrics["n_evaluated"] = float(n_matched)

mlflow.set_experiment(EXPERIMENT)
with mlflow.start_run(run_name="llm_toxicity_classification") as run:
    mlflow.log_metrics(metrics)
    TRUN_ID = run.info.run_id

print(pd.DataFrame([metrics]).T.rename(columns={0:"value"}).round(4))
print("\nClassification report (LLM):")
print(classification_report(pdf["gt_is_toxic"], pdf["is_toxic"], target_names=["clean","toxic"]))